### Bibliotecas e Dependências

In [ ]:
pip install -U pymupdf4llm

In [ ]:
import pandas as pd
import requests
import pymupdf4llm
import re
import os
import json
import unicodedata
import urllib3
from langdetect import detect
import time

Leitura do Arquivo

In [ ]:
df = pd.read_csv(r'C:\Users\05646078199\Projetos\Projeto-Mestrado\Corpus\metadados_completos_sol_sbc.csv')

In [ ]:
df["index"] = range(1, len(df) + 1)

In [ ]:
df.info()

In [ ]:
df.to_csv(r'C:\Users\05646078199\Projetos\Projeto-Mestrado\Corpus\metadados_completos_sol_sbc.csv', index=False)

In [ ]:
df_test = df[
    (df["index"] >= 1001) &
    (df["index"] <= 2000)
]

In [62]:
# OK
df_01 =df[
    (df["index"] >= 2001) &
    (df["index"] <= 5000)
]

#OK
df_02 =df[
    (df["index"] >= 5001) &
    (df["index"] <= 8000)
]

# OK
df_03 =df[
    (df["index"] >= 8001) &
    (df["index"] <= 11000)
]

# OK
df_04 =df[
    (df["index"] >= 11001) &
    (df["index"] <= 14000)
]

# OK
df_05 =df[
    (df["index"] >= 14001) &
    (df["index"] <= 17000)
]

df_extra =df[
    (df["index"] >= 20001) &
    (df["index"] <= 21000)
]

In [63]:
df_extra.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1000 entries, 20000 to 20999
Data columns (total 12 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   Title      1000 non-null   object
 1   Category   1000 non-null   object
 2   URL_Title  1000 non-null   object
 3   Authors    1000 non-null   object
 4   Event      1000 non-null   object
 5   Date       1000 non-null   object
 6   Box        1000 non-null   object
 7   Abstract   1000 non-null   object
 8   Keywords   1000 non-null   object
 9   Publisher  1000 non-null   object
 10  URL_Paper  1000 non-null   object
 11  index      1000 non-null   int64 
dtypes: int64(1), object(11)
memory usage: 101.6+ KB


In [ ]:
"""

1. Requisição para obtenção dos PDF - OK
2. Armazenamento do PDF Localmente - OK
3. Extração do conteúdo do PDF utilizando o pymupdf4llm armazenando o resultado em formato Markdown - OK
4. Identificação das seções e extração do conteúdo de cada seção - OK
5. Normalização do Texto 
6. Detecção do Idioma da Introdução ou Conclusão 
7. Armazenamento do conteúdo extraído em JSON (um arquivo JSON por artigo)

"""

In [ ]:
def detect_article_language(sections):

    # tenta usar introdução primeiro
    for section in sections:

        title = section["title"].lower()

        if "introdu" in title:
            text = section["content"][:2000]

            try:
                return detect(text)
            except:
                pass

    # fallback: usa maior seção do artigo
    try:

        largest_section = max(
            sections,
            key=lambda s: len(s["content"])
        )

        return detect(largest_section["content"][:2000])

    except:
        return "unknown"

In [ ]:
def save_article_json(article_data, output_path):

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(
            article_data,
            f,
            ensure_ascii=False,
            indent=4
        )

In [ ]:
def normalize_text(text):

    # normalização unicode
    text = unicodedata.normalize("NFKC", text)

    # remove hifenização de quebra de linha
    text = re.sub(r'-\s*\n\s*', '', text)

    # transforma quebras simples em espaço
    text = re.sub(r'(?<!\n)\n(?!\n)', ' ', text)

    # remove múltiplos espaços
    text = re.sub(r'\s+', ' ', text)

    # remove espaços nas bordas
    text = text.strip()

    return text

In [ ]:
def extract_sections(md_text):
    pattern = re.compile(r'^(#{1,6})\s+(.*)$', re.MULTILINE)

    matches = list(pattern.finditer(md_text))

    sections = []

    for i, match in enumerate(matches):
        level = len(match.group(1))

        # limpa markdown do título
        title = match.group(2).strip()
        title = re.sub(r'\*+', '', title).strip()

        start = match.end()

        if i + 1 < len(matches):
            end = matches[i + 1].start()
        else:
            end = len(md_text)

        content = md_text[start:end].strip()

        content = normalize_text(content)

        sections.append({
            "level": level,
            "title": title,
            "content": content
        })

    return sections

In [65]:
os.makedirs("pdfs", exist_ok=True)
os.makedirs("markdown", exist_ok=True)
os.makedirs("json", exist_ok=True)

# =========================================================
# CONFIGURAÇÕES
# =========================================================

headers = {
    "User-Agent": "Mozilla/5.0",
    "Accept": "application/pdf"
}

MAX_RETRIES = 1

failed_articles = []

total = len(df_extra)

# =========================================================
# LOOP PRINCIPAL
# =========================================================

for index, row in df_extra.iterrows():

    paper_id = f"sbc_{index}"

    pdf_path = f"pdfs/{paper_id}.pdf"
    md_path = f"markdown/{paper_id}.md"
    json_path = f"json/{paper_id}.json"

    print("\n" + "=" * 80)
    print(f"[{index + 1}/{total}] Processando {paper_id}")

    # =====================================================
    # EVITA REPROCESSAMENTO
    # =====================================================

    if os.path.exists(json_path):

        print("Artigo já processado.")
        continue

    try:

        # =================================================
        # URL PDF
        # =================================================

        url = row["URL_Paper"]

        pdf_url = url.replace("/view/", "/download/")

        print("URL PDF:", pdf_url)

        # =================================================
        # DOWNLOAD COM RETRY
        # =================================================

        response = None

        for attempt in range(MAX_RETRIES):

            try:

                print(f"Tentativa download {attempt + 1}/{MAX_RETRIES}")

                response = requests.get(
                    pdf_url,
                    headers=headers,
                    timeout=60,
                    verify=False
                )

                response.raise_for_status()

                # valida assinatura PDF
                if not response.content.startswith(b"%PDF"):
                    raise Exception("Arquivo baixado não é PDF")

                print("Download realizado com sucesso.")

                break

            except Exception as e:

                print(f"Falha na tentativa {attempt + 1}: {e}")

                if attempt == MAX_RETRIES - 1:
                    raise

                time.sleep(2)

        # =================================================
        # VALIDAÇÕES
        # =================================================

        if response is None:
            raise Exception("Resposta inválida")

        if len(response.content) == 0:
            raise Exception("PDF vazio")

        # =================================================
        # SALVA PDF
        # =================================================

        with open(pdf_path, "wb") as f:
            f.write(response.content)

        print("PDF salvo.")

        # =================================================
        # PDF -> MARKDOWN
        # =================================================

        md_text = pymupdf4llm.to_markdown(
            pdf_path,
            write_images=False
        )

        if not md_text.strip():
            raise Exception("Markdown vazio")

        with open(md_path, "w", encoding="utf-8") as f:
            f.write(md_text)

        print("Markdown salvo.")

        # =================================================
        # EXTRAÇÃO DE SEÇÕES
        # =================================================

        sections = extract_sections(md_text)

        if len(sections) == 0:
            raise Exception("Nenhuma seção encontrada")

        print(f"{len(sections)} seções encontradas.")

        # =================================================
        # DETECÇÃO DE IDIOMA
        # =================================================

        language = detect_article_language(sections)

        print("Idioma detectado:", language)

        # =================================================
        # ESTRUTURA FINAL
        # =================================================

        article_data = {
            "paper_id": paper_id,
            "title": row["Title"],
            "event": row["Event"],
            "authors": row["Authors"],
            "abstract_original": row["Abstract"],
            "url_paper": pdf_url,
            "language": language,
            "status": "success",
            "sections": sections
        }

        # =================================================
        # SALVA JSON
        # =================================================

        save_article_json(article_data, json_path)

        print("JSON salvo.")

        # =================================================
        # LIMPEZA OPCIONAL
        # =================================================

        # remove intermediários
        # descomente se quiser economizar espaço

        # os.remove(pdf_path)
        # os.remove(md_path)

    except Exception as e:

        error_message = str(e)

        print(f"Erro no artigo {paper_id}: {error_message}")

        failed_articles.append({
            "paper_id": paper_id,
            "url": pdf_url,
            "error": error_message
        })

# =========================================================
# SALVA LOG DE FALHAS
# =========================================================

with open("failed_articles.json", "w", encoding="utf-8") as f:

    json.dump(
        failed_articles,
        f,
        ensure_ascii=False,
        indent=4
    )

print("\nProcessamento finalizado.")
print(f"Falhas totais: {len(failed_articles)}")


[20001/1000] Processando sbc_20000
URL PDF: https://sol.sbc.org.br/index.php/sbes/article/download/24239/24062
Tentativa download 1/1
Download realizado com sucesso.
PDF salvo.
Markdown salvo.
Erro no artigo sbc_20000: Nenhuma seção encontrada

[20002/1000] Processando sbc_20001
URL PDF: https://sol.sbc.org.br/index.php/sbes/article/download/21208/21033
Tentativa download 1/1
Download realizado com sucesso.
PDF salvo.
Markdown salvo.
18 seções encontradas.
Idioma detectado: pt
JSON salvo.

[20003/1000] Processando sbc_20002
URL PDF: https://sol.sbc.org.br/index.php/sbes/article/download/24261/24084
Tentativa download 1/1
Download realizado com sucesso.
PDF salvo.
Markdown salvo.
Erro no artigo sbc_20002: Nenhuma seção encontrada

[20004/1000] Processando sbc_20003
URL PDF: https://sol.sbc.org.br/index.php/sbes/article/download/25919/25746
Tentativa download 1/1
Download realizado com sucesso.
PDF salvo.
Markdown salvo.
Erro no artigo sbc_20003: Nenhuma seção encontrada

[20005/1000] P